In [1]:
# Before we write any evaluation code, let's understand the problem

# Imagine your RAG system retrieved these 5 chunks for a question:
retrieved_chunks = [
    "Chunk A — about neural networks",        # relevant
    "Chunk B — about cooking recipes",         # NOT relevant
    "Chunk C — about deep learning",           # relevant
    "Chunk D — about football",                # NOT relevant
    "Chunk E — about backpropagation",         # relevant
]

# Ground truth — which chunks are actually relevant?
relevant_chunks = {"Chunk A", "Chunk C", "Chunk E"}

# ── The naive question: "did it work?" ────────────────────────────────────
retrieved_set = {c.split(" — ")[0] for c in retrieved_chunks}
overlap       = retrieved_set & relevant_chunks

print("Retrieved:", retrieved_set)
print("Relevant: ", relevant_chunks)
print("Overlap:  ", overlap)
print()
print("Did it work?", "Yes" if overlap else "No")
print()
print("--- The problem ---")
print("'Yes it worked' tells you nothing useful.")
print("How many relevant chunks did you miss?")
print("Did the best chunk come back first or last?")
print("Would you trust this system in production based on 'yes/no'?")
print()
print("This is why retrieval evaluation metrics exist.")

Retrieved: {'Chunk C', 'Chunk B', 'Chunk E', 'Chunk D', 'Chunk A'}
Relevant:  {'Chunk C', 'Chunk E', 'Chunk A'}
Overlap:   {'Chunk C', 'Chunk E', 'Chunk A'}

Did it work? Yes

--- The problem ---
'Yes it worked' tells you nothing useful.
How many relevant chunks did you miss?
Did the best chunk come back first or last?
Would you trust this system in production based on 'yes/no'?

This is why retrieval evaluation metrics exist.


In [1]:
def precision_at_k(retrieved: list, relevant: set, k:int) -> float:
    """
    Of the top-K chunks retrieved, what fraction are relevant?
    Precision@K = |relevant ∩ retrieved[:K]| / K
    """

    retrieved_at_k = set(retrieved[:k])
    hits = len(retrieved_at_k & relevant)
    return hits/k

def recall_at_k(retrieved: list, relevant: set, k:int) -> float:

    if not relevant:
        return 0.0
    retrieved_at_k = set(retrieved[:k])
    hits = len(retrieved_at_k & relevant)
    return hits/len(relevant)

# ── Test with our example ──────────────────────────────────────────────────
retrieved = ["Chunk A", "Chunk B", "Chunk C", "Chunk D", "Chunk E"]
relevant  = {"Chunk A", "Chunk C", "Chunk E"}

print("Retrieved order:", retrieved)
print("Relevant chunks:", relevant)
print()

for k in [1, 2, 3, 4, 5]:
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    retrieved_display = retrieved[:k]
    print(f"K={k}  Retrieved={retrieved_display}")
    print(f"      Precision@{k} = {p:.2f}  |  Recall@{k} = {r:.2f}")
    print()

print("─" * 60)
print("Key insight:")
print("Precision and Recall trade off against each other.")
print("Retrieve more chunks → Recall goes up, Precision goes down.")
print("The right K depends on your use case.")

Retrieved order: ['Chunk A', 'Chunk B', 'Chunk C', 'Chunk D', 'Chunk E']
Relevant chunks: {'Chunk C', 'Chunk A', 'Chunk E'}

K=1  Retrieved=['Chunk A']
      Precision@1 = 1.00  |  Recall@1 = 0.33

K=2  Retrieved=['Chunk A', 'Chunk B']
      Precision@2 = 0.50  |  Recall@2 = 0.33

K=3  Retrieved=['Chunk A', 'Chunk B', 'Chunk C']
      Precision@3 = 0.67  |  Recall@3 = 0.67

K=4  Retrieved=['Chunk A', 'Chunk B', 'Chunk C', 'Chunk D']
      Precision@4 = 0.50  |  Recall@4 = 0.67

K=5  Retrieved=['Chunk A', 'Chunk B', 'Chunk C', 'Chunk D', 'Chunk E']
      Precision@5 = 0.60  |  Recall@5 = 1.00

────────────────────────────────────────────────────────────
Key insight:
Precision and Recall trade off against each other.
Retrieve more chunks → Recall goes up, Precision goes down.
The right K depends on your use case.


In [2]:
def hit_rate_at_k(retrieved: list, relevant: set, k:int) -> float:
    """
    Did at least one relevant chunk appear in the top K?
    Binary — 1.0 or 0.0. Useful for question answering where
    you just need ONE good chunk to answer correctly.
    Hit Rate@K = 1 if any(retrieved[:K]) in relevant else 0
    """

    return 1.0 if set(retrieved[:k]) & relevant else 0.0

def mean_reciprocal_rank(retrieved: list, relevant: set) -> float:
    """
    Where does the FIRST relevant chunk appear in the ranking?
    MRR = 1 / rank_of_first_relevant_chunk
    
    If first relevant chunk is at position 1 → MRR = 1.0 (perfect)
    If first relevant chunk is at position 2 → MRR = 0.5
    If first relevant chunk is at position 5 → MRR = 0.2
    If no relevant chunk found              → MRR = 0.0
    """

    for rank, chunk in enumerate(retrieved, start=1):
        if chunk in relevant:
            return 1.0/rank
    return 0.0

# ── Compare three retrieval orderings — same chunks, different ranks ───────
relevant = {"Chunk A", "Chunk C", "Chunk E"}

scenarios = {
    "Best case  — relevant chunks ranked first": 
        ["Chunk A", "Chunk C", "Chunk E", "Chunk B", "Chunk D"],
    "Worst case — relevant chunks ranked last":  
        ["Chunk B", "Chunk D", "Chunk A", "Chunk C", "Chunk E"],
    "Mixed case — one relevant chunk first":     
        ["Chunk A", "Chunk B", "Chunk D", "Chunk C", "Chunk E"],
}


print(f"{'Scenario':<45} {'HR@3':>6} {'HR@5':>6} {'MRR':>6} {'P@3':>6} {'R@3':>6}")
print("─" * 75)

for name, retrieved in scenarios.items():
    hr3 = hit_rate_at_k(retrieved, relevant, k=3)
    hr5 = hit_rate_at_k(retrieved, relevant, k=5)
    mrr = mean_reciprocal_rank(retrieved, relevant)
    p3  = precision_at_k(retrieved, relevant, k=3)
    r3  = recall_at_k(retrieved, relevant, k=3)
    print(f"{name:<45} {hr3:>6.2f} {hr5:>6.2f} {mrr:>6.2f} {p3:>6.2f} {r3:>6.2f}")

print()
print("Key insight:")
print("Precision@3 and Recall@3 are identical for best and worst case.")
print("MRR reveals what they miss — the RANKING matters, not just presence.")
print()
print("For RAG: a relevant chunk at rank 1 is worth more than at rank 5")
print("because LLMs pay more attention to context at the top.")

Scenario                                        HR@3   HR@5    MRR    P@3    R@3
───────────────────────────────────────────────────────────────────────────
Best case  — relevant chunks ranked first       1.00   1.00   1.00   1.00   1.00
Worst case — relevant chunks ranked last        1.00   1.00   0.33   0.33   0.33
Mixed case — one relevant chunk first           1.00   1.00   1.00   0.33   0.33

Key insight:
Precision@3 and Recall@3 are identical for best and worst case.
MRR reveals what they miss — the RANKING matters, not just presence.

For RAG: a relevant chunk at rank 1 is worth more than at rank 5
because LLMs pay more attention to context at the top.


In [3]:
# NORMALIZED DISCOUNTED CUMULATIVE GAIN - NDCG

import numpy as np

def dcg_at_k(retrieved: list, relevance_scores: dict, k:int) -> float:
    """
    Discounted Cumulative Gain — rewards relevant chunks ranked higher.
    
    relevance_scores: dict mapping chunk → relevance score
    2 = highly relevant, 1 = partially relevant, 0 = not relevant
    
    DCG@K = sum(relevance[i] / log2(rank + 1)) for rank 1..K
    
    The log2 discount means rank 1 contributes full value,
    rank 2 contributes half, rank 3 contributes less, etc.
    """
    
    dcg = 0.0
    for rank, chunk in enumerate(retrieved[:k], start = 1):
        rel = relevance_scores.get(chunk, 0)
        dcg += rel / np.log2(rank + 1)
    return dcg

def ndcg_at_k(retrieved: list, relevance_scores: dict, k: int) -> float:
    """
    Normalized DCG — divides by the ideal DCG (perfect ranking).
    Score between 0 and 1. 1.0 = perfect ranking.
    """

    actual_dcg = dcg_at_k(retrieved, relevance_scores, k)

    # Ideal ranking - sort all chunks by relevance descending
    ideal_order = sorted(
        relevance_scores.keys(), 
        key = lambda x: relevance_scores[x], 
        reverse=True
    )

    ideal_dcg = dcg_at_k(ideal_order, relevance_scores, k)

    return actual_dcg/ideal_dcg if ideal_dcg > 0 else 0.0

# ── Example with graded relevance ─────────────────────────────────────────
# Binary relevance: relevant or not
# Graded relevance: highly relevant, partially relevant, not relevant
relevance_scores = {
    "Chunk A": 2,   # highly relevant — directly answers the question
    "Chunk B": 0,   # not relevant
    "Chunk C": 1,   # partially relevant — related but indirect
    "Chunk D": 0,   # not relevant
    "Chunk E": 2,   # highly relevant — directly answers the question
}

scenarios = {
    "Perfect ranking":   ["Chunk A", "Chunk E", "Chunk C", "Chunk B", "Chunk D"],
    "Good ranking":      ["Chunk A", "Chunk C", "Chunk E", "Chunk B", "Chunk D"],
    "Bad ranking":       ["Chunk B", "Chunk D", "Chunk C", "Chunk A", "Chunk E"],
    "Worst ranking":     ["Chunk B", "Chunk D", "Chunk C", "Chunk E", "Chunk A"],
}

print(f"{'Scenario':<25} {'NDCG@1':>8} {'NDCG@3':>8} {'NDCG@5':>8} {'MRR':>8}")
print("─" * 55)

relevant_set = {c for c, s in relevance_scores.items() if s > 0}
for name, retrieved in scenarios.items():
    n1  = ndcg_at_k(retrieved, relevance_scores, k=1)
    n3  = ndcg_at_k(retrieved, relevance_scores, k=3)
    n5  = ndcg_at_k(retrieved, relevance_scores, k=5)
    mrr = mean_reciprocal_rank(retrieved, relevant_set)
    print(f"{name:<25} {n1:>8.3f} {n3:>8.3f} {n5:>8.3f} {mrr:>8.3f}")

print()
print("Key insight:")
print("NDCG distinguishes between 'partially relevant' and 'highly relevant'.")
print("MRR and Precision treat all relevant chunks equally.")
print("NDCG rewards systems that rank the BEST chunks highest.")
print()
print("Discount factors by rank:")
for rank in range(1, 6):
    discount = 1 / np.log2(rank + 1)
    bar      = "█" * int(discount * 20)
    print(f"  Rank {rank}: {discount:.3f} {bar}")

    

Scenario                    NDCG@1   NDCG@3   NDCG@5      MRR
───────────────────────────────────────────────────────
Perfect ranking              1.000    1.000    1.000    1.000
Good ranking                 1.000    0.965    0.965    1.000
Bad ranking                  0.000    0.133    0.568    0.333
Worst ranking                0.000    0.133    0.568    0.333

Key insight:
NDCG distinguishes between 'partially relevant' and 'highly relevant'.
MRR and Precision treat all relevant chunks equally.
NDCG rewards systems that rank the BEST chunks highest.

Discount factors by rank:
  Rank 1: 1.000 ████████████████████
  Rank 2: 0.631 ████████████
  Rank 3: 0.500 ██████████
  Rank 4: 0.431 ████████
  Rank 5: 0.387 ███████


In [4]:
# ANSWER QUALITY METRICS

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.2")
parser = StrOutputParser()

# ── Three answer quality metrics ───────────────────────────────────────────
# We use the LLM itself as a judge — "LLM-as-judge" pattern

faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an evaluation assistant. Your job is to check if an answer 
is faithful to the provided context — meaning it only uses information from the context
and does not add outside knowledge or hallucinate.

Respond with ONLY a JSON object like this:
{{"score": 0.9, "reason": "one sentence explanation"}}

Score 1.0 = completely faithful, every claim supported by context
Score 0.5 = partially faithful, some claims unsupported  
Score 0.0 = completely unfaithful, answer ignores or contradicts context"""),
    ("human", """Context:
{context}

Answer:
{answer}

Is this answer faithful to the context?""")
])

relevance_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an evaluation assistant. Your job is to check if an answer
actually addresses the question asked.

Respond with ONLY a JSON object like this:
{{"score": 0.9, "reason": "one sentence explanation"}}

Score 1.0 = answer directly and completely addresses the question
Score 0.5 = answer partially addresses the question
Score 0.0 = answer does not address the question at all"""),
    ("human", """Question:
{question}

Answer:
{answer}

Does this answer address the question?""")
])

faithfulness_chain = faithfulness_prompt | llm | parser
relevance_chain = relevance_prompt | llm | parser

import json
import re

def parse_score(response: str) -> dict:
    """Extract score and reason from LLM judge response"""
    try:
        # Find JSON in response
        match = re.search(r'\{.*?\}', response, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {"score": 0.0, "reason": "Could not parse response"}

def evaluate_answer(question: str, context: str, answer: str) -> dict:
    """Run all answer quality metrics"""

    faith_response = faithfulness_chain.invoke({
        "context" : context, 
        "answer" : answer
    })
    rel_response = relevance_chain.invoke({
        "question" : question, 
        "answer" : answer
    })

    faith = parse_score(faith_response)
    rel = parse_score(rel_response)

    return {
        "faithfulness" : faith.get("score", 0.0),
        "faithfulness_reason" : faith.get("reason", ""),
        "answer_relevance" : rel.get("score", 0.0),
        "relevance_reason" : rel.get("reason", ""),
    }

# ── Test with three answer types ───────────────────────────────────────────
context = """Machine learning is a subset of artificial intelligence that enables 
systems to learn from data without being explicitly programmed. 
Supervised learning uses labeled training data to learn mappings from inputs to outputs.
Common supervised algorithms include linear regression and decision trees."""

question = "What is supervised learning?"

answers = {
    "Good answer — grounded in context":
        "Supervised learning uses labeled training data to learn mappings from inputs to outputs.",
    "Hallucinated answer — adds outside info":
        "Supervised learning was invented by Arthur Samuel in 1959 and uses neural networks to classify images.",
    "Irrelevant answer — doesn't address question":
        "Machine learning is a very interesting field with many applications in industry.",
}

print(f"Context: {context[:100]}...\n")
print(f"Question: {question}\n")
print("─" * 70)

for answer_type, answer in answers.items():
    print(f"\nAnswer type: {answer_type}")
    print(f"Answer: {answer[:80]}...")
    scores = evaluate_answer(question, context, answer)
    print(f"Faithfulness:     {scores['faithfulness']:.2f} — {scores['faithfulness_reason']}")
    print(f"Answer Relevance: {scores['answer_relevance']:.2f} — {scores['relevance_reason']}")


Context: Machine learning is a subset of artificial intelligence that enables 
systems to learn from data wit...

Question: What is supervised learning?

──────────────────────────────────────────────────────────────────────

Answer type: Good answer — grounded in context
Answer: Supervised learning uses labeled training data to learn mappings from inputs to ...
Faithfulness:     1.00 — entire statement is supported by context, correctly defining supervised learning
Answer Relevance: 1.00 — The answer directly and completely defines supervised learning, covering its key characteristic of using labeled training data.

Answer type: Hallucinated answer — adds outside info
Answer: Supervised learning was invented by Arthur Samuel in 1959 and uses neural networ...
Faithfulness:     0.50 — claims that supervised learning was invented by Arthur Samuel and uses neural networks, which are not supported by the context
Answer Relevance: 0.00 — the answer does not define what supervised learning i

In [5]:
# LATENCY METRICS

import time
import statistics

def measure_latency(func, *args, n_runs=3, **kwargs) -> dict:
    """Run a function n times and return latency statistics"""
    latencies = []
    result = None
    for _ in range(n_runs):
        start = time.time()
        result = func(*args, **kwargs)
        latencies.append(time.time() - start)

    return{
        "result" : result,
        "mean":   round(statistics.mean(latencies), 3),
        "min":    round(min(latencies), 3),
        "max":    round(max(latencies), 3),
        "std":    round(statistics.stdev(latencies), 3) if len(latencies) > 1 else 0.0,

    }

# Simulate retrieval and generation latency

import sys
sys.path.append("..")
from src.rag import RAGPipeline

print ("Loading RAG pipeline")
rag = RAGPipeline()
rag.load_pdf("../data/Fundamentals of Data Engineering.pdf")
print(f"Loaded - {len(rag.chunks)} chunks indexed \n")

# Measure retrieval latency
def retrieval_only(question, k=3):
    """Just the FAISS search - no LLM"""
    import numpy as np
    query_emb = rag.embedding_model.encode([question]).astype(np.float32)
    distances, indices = rag.index.search(query_emb, k)
    return [rag.chunks[i] for i in indices[0]]


# Measure full RAG latency
def full_rag(question):
    return rag.query(question)

questions = [
    "What is data engineering?",
    "Explain the key concept of monolith vs modular",
    "What are principles of good architecture?",
]

print(f"{'Metric':<30} {'Mean (s)':>10} {'Min (s)':>10} {'Max (s)':>10}")
print("─" * 65)


retrieval_times = []
full_times      = []

for question in questions:
    r_stats = measure_latency(retrieval_only, question, n_runs=3)
    f_stats = measure_latency(full_rag,       question, n_runs=2)

    retrieval_times.append(r_stats["mean"])
    full_times.append(f_stats["mean"])

    print(f"\nQuestion: {question[:50]}")
    print(f"  {'Retrieval only':<28} {r_stats['mean']:>10.3f} {r_stats['min']:>10.3f} {r_stats['max']:>10.3f}")
    print(f"  {'Full RAG (retrieval + LLM)':<28} {f_stats['mean']:>10.3f} {f_stats['min']:>10.3f} {f_stats['max']:>10.3f}")
    print(f"  LLM accounts for: {((f_stats['mean'] - r_stats['mean']) / f_stats['mean'] * 100):.1f}% of total latency")

print("\n" + "─" * 65)
print(f"Average retrieval latency: {statistics.mean(retrieval_times):.3f}s")
print(f"Average full RAG latency:  {statistics.mean(full_times):.3f}s")
print(f"\nKey insight: FAISS retrieval is near-instant.")
print(f"The LLM generation dominates total latency.")
print(f"To speed up RAG: optimize the LLM, not the retrieval.")


Loading RAG pipeline


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5237.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded - 1416 chunks indexed 

Metric                           Mean (s)    Min (s)    Max (s)
─────────────────────────────────────────────────────────────────

Question: What is data engineering?
  Retrieval only                    0.010      0.008      0.012
  Full RAG (retrieval + LLM)        4.370      0.620      8.121
  LLM accounts for: 99.8% of total latency

Question: Explain the key concept of monolith vs modular
  Retrieval only                    0.009      0.008      0.010
  Full RAG (retrieval + LLM)        1.484      1.223      1.746
  LLM accounts for: 99.4% of total latency

Question: What are principles of good architecture?
  Retrieval only                    0.009      0.008      0.009
  Full RAG (retrieval + LLM)        1.540      0.826      2.254
  LLM accounts for: 99.4% of total latency

─────────────────────────────────────────────────────────────────
Average retrieval latency: 0.009s
Average full RAG latency:  2.465s

Key insight: FAISS retrieval is near-insta